<a href="https://colab.research.google.com/github/QuintonPang/Fine_Tuning_DistilBERT_for_IMDb_with_Sequence_Classification/blob/main/Fine_Tuning_DistilBERT_for_IMDb_with_Sequence_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
! pip install transformers datasets evaluate accelerate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.9 MB/s eta 0:00:00


In [2]:
from huggingface_hub import notebook_login
notebook_login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [3]:
from datasets import load_dataset

imdb = load_dataset("stanfordnlp/imdb")

README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [4]:
imdb["test"][0] # test set

{'text': 'I love sci-fi and am willing to put up with a lot. Sci-fi movies/TV are usually underfunded, under-appreciated and misunderstood. I tried to like this, I really did, but it is to good TV sci-fi as Babylon 5 is to Star Trek (the original). Silly prosthetics, cheap cardboard sets, stilted dialogues, CG that doesn\'t match the background, and painfully one-dimensional characters cannot be overcome with a \'sci-fi\' setting. (I\'m sure there are those of you out there who think Babylon 5 is good sci-fi TV. It\'s not. It\'s clichéd and uninspiring.) While US viewers might like emotion and character development, sci-fi is a genre that does not take itself seriously (cf. Star Trek). It may treat important issues, yet not as a serious philosophy. It\'s really difficult to care about the characters here as they are not simply foolish, just missing a spark of life. Their actions and reactions are wooden and predictable, often painful to watch. The makers of Earth KNOW it\'s rubbish as 

In [5]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [6]:
# breaks down text into tokens
def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True)

In [7]:
tokenized_imdb = imdb.map(preprocess_function, batched=True)

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [8]:
# pad sentence to longest length as GPU cannot process different lengths at same time in same batch
# a batch is a matrix

from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [9]:
import evaluate

accuracy = evaluate.load("accuracy")

In [10]:
import numpy as np


def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy.compute(predictions=predictions, references=labels)

In [11]:
id2label = {0: "NEGATIVE", 1: "POSITIVE"}
label2id = {"NEGATIVE": 0, "POSITIVE": 1}

In [12]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

# fine tunining this model

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert/distilbert-base-uncased", num_labels=2, id2label=id2label, label2id=label2id
)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [13]:
training_args = TrainingArguments(
    output_dir="my_awesome_model",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    push_to_hub=True,
)


In [15]:
# fine tunining this model

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_imdb["train"],
    eval_dataset=tokenized_imdb["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [16]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.225678,0.202239,0.923160
2,0.147079,0.234786,0.929960


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3126, training_loss=0.20718690575656415, metrics={'train_runtime': 3546.0226, 'train_samples_per_second': 14.1, 'train_steps_per_second': 0.882, 'total_flos': 6556904415524352.0, 'train_loss': 0.20718690575656415, 'epoch': 2.0})

In [17]:
# share to hugging face
trainer.push_to_hub()

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...e_model/model.safetensors:  57%|#####6    |  152MB /  268MB            

  ...e_model/training_args.bin: 100%|##########| 5.26kB / 5.26kB            

CommitInfo(commit_url='https://huggingface.co/quintonpyx/my_awesome_model/commit/baef8a4b111e809f1406614b423d3cf150e902cb', commit_message='End of training', commit_description='', oid='baef8a4b111e809f1406614b423d3cf150e902cb', pr_url=None, repo_url=RepoUrl('https://huggingface.co/quintonpyx/my_awesome_model', endpoint='https://huggingface.co', repo_type='model', repo_id='quintonpyx/my_awesome_model'), pr_revision=None, pr_num=None)

In [18]:
# use pipeline to perform sentiment analysis
text = "I love this course"

In [19]:
from transformers import pipeline

classifier = pipeline("sentiment-analysis", model="quintonpyx/my_awesome_model")
classifier(text)

config.json:   0%|          | 0.00/783 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/351 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

[{'label': 'POSITIVE', 'score': 0.98204505443573}]

In [21]:
# manual way
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("quintonpyx/my_awesome_model")
inputs = tokenizer(text, return_tensors="pt") # Convert the output directly into PyTorch Tensors
inputs

{'input_ids': tensor([[ 101, 1045, 2293, 2023, 2607,  102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1]])}

In [25]:
from transformers import AutoModelForSequenceClassification
import torch

model = AutoModelForSequenceClassification.from_pretrained("quintonpyx/my_awesome_model")
with torch.no_grad(): # no change in weights
    logits = model(**inputs).logits # The raw, unnormalized outputs generated by the very last layer of the neural network.

logits

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tensor([[-1.8772,  2.1245]])

In [26]:
predicted_class_id = logits.argmax().item() # This line figures out which class index the model is most confident about.
model.config.id2label[predicted_class_id] # maps that index integer back to an actual text string using the model's internal dictionary.

'POSITIVE'

In [29]:
inputs_2 = tokenizer("FUCK", return_tensors="pt") # Convert the output directly into PyTorch Tensors
inputs_2

{'input_ids': tensor([[ 101, 6616,  102]]), 'token_type_ids': tensor([[0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1]])}

In [30]:
with torch.no_grad(): # no change in weights
    logits_2 = model(**inputs_2).logits # The raw, unnormalized outputs generated by the very last layer of the neural network.

logits_2

tensor([[ 0.4731, -0.4682]])

In [33]:
predicted = logits_2.argmax().item()
model.config.id2label[predicted]

'NEGATIVE'

In [34]:
# Test with some custom test strings!
test_reviews = [
    "This movie was an absolute masterpiece. The acting was incredible!",
    "Strictly terrible. A complete waste of two hours of my life.",
    "I thought it would be bad, but it actually turned out to be amazing.",
    "Definite Oscar contender if the category was 'Most Boring Film Ever'." # Sarcasm test
]

results = classifier(test_reviews)

for review, result in zip(test_reviews, results):
    print(f"Review: {review}")
    print(f"Prediction: {result['label']} (Confidence: {result['score']:.4f})\n")

Review: This movie was an absolute masterpiece. The acting was incredible!
Prediction: POSITIVE (Confidence: 0.9919)

Review: Strictly terrible. A complete waste of two hours of my life.
Prediction: NEGATIVE (Confidence: 0.9909)

Review: I thought it would be bad, but it actually turned out to be amazing.
Prediction: POSITIVE (Confidence: 0.9297)

Review: Definite Oscar contender if the category was 'Most Boring Film Ever'.
Prediction: NEGATIVE (Confidence: 0.9181)

